In [ ]:
import pandas as pd
from sktime.datasets import load_gunpoint
from sktime.clustering.k_means import TimeSeriesKMeans
import seaborn as sns
from sktime.utils.plotting import plot_series
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Clustering

In [ ]:
X_train, y_train = load_gunpoint(split="TRAIN", return_type="numpy3D")
X_train.shape, y_train.shape

In [ ]:
plt.title("A time series from the GunPoint dataset")
plt.plot(X_train[0].T)

In [ ]:
df = pd.DataFrame(np.squeeze(X_train))
df_melt = df.melt(value_name="observation", var_name="time")
sns.lineplot(df_melt, x="time", y="observation", estimator=None)
plt.title("All time series from the GunPoint dataset")
plt.show()

In [ ]:
sns.lineplot(df_melt, x="time", y="observation", estimator="mean")
plt.title("All time series from the GunPoint dataset (mean and std)")
plt.show()

## Distance-based Clustering

### Euclidean

In [ ]:
clusterer = TimeSeriesKMeans(n_clusters=2, metric="euclidean")

In [ ]:
%%time
clusterer.fit(X_train)

In [ ]:
df["cluster"] = clusterer.labels_
df.head()

In [ ]:
df_melt = df.melt(value_name="observation", var_name="time", id_vars="cluster")
sns.lineplot(df_melt, x="time", y="observation", hue="cluster")
plt.title("Clustered time series dataset (mean and std)")
plt.show()

In [ ]:
sns.lineplot(df_melt, x="time", y="observation", hue="cluster", estimator=None)
plt.title("Clustered time series dataset")
plt.show()

In [ ]:
pd.crosstab(df["cluster"], y_train).plot(kind="bar", stacked=True)
plt.legend(title="class")

### DTW

In [ ]:
clusterer = TimeSeriesKMeans(n_clusters=2, metric="dtw",
                             distance_params={'itakura_max_slope': 10})

In [ ]:
%%time
clusterer.fit(X_train)

In [ ]:
df["cluster"] = clusterer.labels_
df.head()

In [ ]:
df_melt = df.melt(value_name="observation", var_name="time", id_vars="cluster")
sns.lineplot(df_melt, x="time", y="observation", hue="cluster")
plt.title("Clustered time series dataset (mean and std)")
plt.show()

In [ ]:
sns.lineplot(df_melt, x="time", y="observation", hue="cluster", estimator=None)
plt.title("Clustered time series dataset")
plt.show()

In [ ]:
pd.crosstab(df["cluster"], y_train).plot(kind="bar", stacked=True)
plt.legend(title="class")

## Features-based Clustering

Custom Features Extraction

In [ ]:
import scipy.stats as stats

In [ ]:
def calculate_features(values):
    features = {
        'avg': np.mean(values),
        'std': np.std(values),
        'var': np.var(values),
        'med': np.median(values),
        '10p': np.percentile(values, 10),
        '25p': np.percentile(values, 25),
        '50p': np.percentile(values, 50),
        '75p': np.percentile(values, 75),
        '90p': np.percentile(values, 90),
        'iqr': np.percentile(values, 75) - np.percentile(values, 25),
        'cov': 1.0 * np.mean(values) / np.std(values),
        'skw': stats.skew(values),
        'kur': stats.kurtosis(values)
    }

    return features

In [ ]:
X_train.shape

In [ ]:
calculate_features(X_train[0][0])

In [ ]:
feature_values = [list(calculate_features(row).values())
                  for row in X_train[:, 0, :]]
index_values = list(range(0, X_train.shape[0]))
column_values = list(calculate_features(X_train[0][0]).keys())

CF_Train = pd.DataFrame(data=feature_values,
                        index=index_values, columns=column_values)
CF_Train

Summary Transformer Extractor

In [ ]:
from sktime.transformations.series.summarize import SummaryTransformer
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

In [ ]:
transformer = SummaryTransformer()

In [ ]:
F_train = transformer.fit_transform(X_train)
F_train

TS-Fresh Feature Extractor

In [ ]:
from sktime.transformations.panel.tsfresh import TSFreshFeatureExtractor

In [ ]:
ts_eff = TSFreshFeatureExtractor(
    default_fc_parameters="comprehensive", show_warnings=False, disable_progressbar=True)

In [ ]:
X_transform1 = ts_eff.fit_transform(X_train)
X_transform1

In [ ]:
clusterer = DBSCAN(eps=0.5, min_samples=5)

In [ ]:
clusterer.fit(F_train)

In [ ]:
F_train["cluster"] = clusterer.labels_

In [ ]:
pca = PCA(n_components=2)
F_train_pca = pca.fit_transform(F_train)
F_train = pd.concat(
    [F_train, pd.DataFrame(F_train_pca, columns=["pca1", "pca2"])], axis=1
)

In [ ]:
sns.scatterplot(F_train, x="pca1", y="pca2", hue="cluster")
plt.show()

In [ ]:
df["cluster"] = clusterer.labels_

In [ ]:
df_melt = df.melt(value_name="observation", var_name="time", id_vars="cluster")
sns.lineplot(df_melt, x="time", y="observation", hue="cluster")
plt.title("Clustered time series dataset (mean and std)")
plt.show()

In [ ]:
sns.lineplot(df_melt, x="time", y="observation", hue="cluster", estimator=None)
plt.title("Clustered time series dataset")
plt.show()

In [ ]:
pd.crosstab(df["cluster"], y_train).plot(kind="bar", stacked=True)

# Approximated Clustering

In [ ]:
from sktime.transformations.panel.dictionary_based import PAA

In [ ]:
transformer = PAA(num_intervals=10)

In [ ]:
X_paa = transformer.fit_transform(X_train)
df_paa = pd.DataFrame(np.squeeze(X_paa))

In [ ]:
plt.plot(np.squeeze(X_paa.T), color="C1")
plt.title("PAA approximation of the time series dataset")
plt.show()

In [ ]:
clusterer = TimeSeriesKMeans(n_clusters=4, metric="dtw")

In [ ]:
%%time
clusterer.fit(X_paa)

In [ ]:
df["cluster"] = clusterer.labels_
df_paa["cluster"] = clusterer.labels_

In [ ]:
df_melt_paa = df_paa.melt(value_name="observation",
                          var_name="time", id_vars="cluster")
sns.lineplot(df_melt_paa, x="time", y="observation", hue="cluster")
plt.title("Clustered approximated time series dataset (mean and std)")

In [ ]:
df_melt = df.melt(value_name="observation", var_name="time", id_vars="cluster")
sns.lineplot(df_melt, x="time", y="observation", hue="cluster")
plt.title("Clustered time series dataset (mean and std)")
plt.show()